In [5]:
import os
import numpy as np
import pandas as pd

CLIENT_FEATURES_PATH = "client_level_features.parquet"

def build_client_features(df: pd.DataFrame, frauds: list[int]) -> pd.DataFrame:
    df = df.sort_values(["person_id", "trn_date"]).copy()

    client_data = df.groupby('person_id').agg(
        num_of_trn=('trn_id', 'nunique'),
        days_visits=('date', 'nunique'),

        num_of_waiters=('waiter_id', 'nunique'),
        num_of_places = ('place_id', 'nunique'),

        gross_amount_mean=('gross_amount', 'mean'),
        gross_amount_sum=('gross_amount', 'sum'),
        gross_amount_max=('gross_amount', 'max'),

        bonuses_accum_sum=('bonusses_accum', 'sum'),
        bonuses_used_sum=('bonusses_used', 'sum'),
        bonus_trn_count=('bonusses_used', lambda x: (x > 0).sum()),

        first_last_trn_diff=(
            'trn_date',
            lambda x: (x.max() - x.min()).total_seconds() / 3600 if len(x) > 1 else np.nan
        ),
        first_second_trn_diff=(
            'trn_date',
            lambda x: (x.sort_values().iloc[1] - x.min()).total_seconds() / 3600 if len(x) > 1 else np.nan
        ),
        first_third_trn_diff=(
            'trn_date',
            lambda x: (x.sort_values().iloc[2] - x.min()).total_seconds() / 3600 if len(x) > 2 else np.nan
        ),

        time_between_trn_median=('time_since_prev_trn_hours', 'median'),

        share_bonus_after_first=(
            'bonus_used_flag',
            lambda x: x[df.loc[x.index, 'trn_count_before'] > 0].mean()
        )
    ).reset_index()

    # derived
    client_data['is_fraud'] = client_data['person_id'].isin(frauds)
    client_data['trn_per_day'] = client_data['num_of_trn'] / client_data['days_visits']
    client_data['share_bonus_trn'] = client_data['bonus_trn_count'] / client_data['num_of_trn']

    # top waiter share
    waiter_share = (
        df.groupby(["person_id", "waiter_id"])
          .size()
          .reset_index(name="cnt")
    )
    top_waiter_share = (
        waiter_share.sort_values(["person_id", "cnt"], ascending=[True, False])
        .groupby("person_id")
        .first()
        .reset_index()
        .rename(columns={"waiter_id": "top_waiter_id"})
    )

    client_data = client_data.merge(
        top_waiter_share[["person_id", "top_waiter_id", "cnt"]],
        on="person_id",
        how="left"
    )

    client_data["share_top_waiter"] = client_data["cnt"] / client_data["num_of_trn"]
    client_data = client_data.drop(columns=["cnt"])

    # share of bonus transactions in top waiter (fraction of trns with bonus used, not sum of bonuses)
    top_waiter_bonus_trn = (
        df.merge(
            top_waiter_share[["person_id", "top_waiter_id"]],
            on="person_id",
            how="left"
        )
        .loc[lambda x: x["waiter_id"] == x["top_waiter_id"]]
        .groupby("person_id", as_index=False)["bonus_used_flag"]
        .sum()
        .rename(columns={"bonus_used_flag": "bonus_trn_count_top_waiter"})
    )

    top_waiter_trn_count = (
        df.merge(
            top_waiter_share[["person_id", "top_waiter_id"]],
            on="person_id",
            how="left"
        )
        .loc[lambda x: x["waiter_id"] == x["top_waiter_id"]]
        .groupby("person_id", as_index=False)
        .size()
        .rename(columns={"size": "trn_count_top_waiter"})
    )

    client_data = client_data.merge(
        top_waiter_bonus_trn,
        on="person_id",
        how="left"
    ).merge(
        top_waiter_trn_count,
        on="person_id",
        how="left"
    )

    client_data["bonus_trn_count_top_waiter"] = client_data["bonus_trn_count_top_waiter"].fillna(0)
    client_data["trn_count_top_waiter"] = client_data["trn_count_top_waiter"].fillna(0)

    client_data["share_bonuses_used_top_waiter"] = np.where(
        client_data["trn_count_top_waiter"] > 0,
        client_data["bonus_trn_count_top_waiter"] / client_data['bonus_trn_count'],
        np.nan
    )

    client_data = client_data.drop(columns=["top_waiter_id", "bonus_trn_count_top_waiter", "trn_count_top_waiter"])

    
    #top place share
    place_share = (
        df.groupby(['person_id', 'place_id'])
        .size()
        .reset_index(name="cnt")
    )
    top_place_share = (
        place_share.sort_values(["person_id", "cnt"], ascending=[True, False])
        .groupby("person_id")
        .first()
        .reset_index()
    )

    client_data = client_data.merge(
        top_place_share[["person_id", "cnt"]],
        on="person_id",
        how="left"
    )

    client_data["share_top_places"] = client_data["cnt"] / client_data["num_of_trn"]
    client_data = client_data.drop(columns=["cnt"])

    return client_data

df = pd.read_parquet("/Users/yuliia/Documents/Fraud-Detection/parquet/processed_transactions.parquet", engine="pyarrow")
frauds = [
    11968000,
    11970409,
    11726701,
    14827913,
    12411311,
    11098795,
    12412748,
    11812494,
    12396334,
    11699175,
    10896313,
    16376555,
    16921722, 
    16846666,
    16794470,
    16440373,
    11766135,
    13969910,
    12234377,
    12861171, 
    13076915, 
    13337997, 
    13239788,
    12614443,
    12646735,  # hz
    13119461,
    13859241,
    13119415,
    3349318,
    12830394,
    12878021
]

if os.path.exists(CLIENT_FEATURES_PATH):
    client_data = pd.read_parquet(CLIENT_FEATURES_PATH)
else:
    client_data = build_client_features(df, frauds)
    client_data.to_parquet(CLIENT_FEATURES_PATH, index=False, engine="pyarrow")